# Lab Prático — TADs em Java: Pilha, Fila e Union-Find

**Disciplina:** COMP0497 — Algoritmos e Estruturas de Dados I

**Professor:** Prof. Dr. André Yoshiaki Kashiwabara — DCOMP/UFS

**E-mail:** andreyoshiaki@dcomp.ufs.br

**Duração estimada:** 90 min

**Pré-requisitos:** Aula sobre TADs, Pilhas, Filas e Union-Find

---

## Objetivos

Ao final deste laboratório, você será capaz de:

1. Implementar a interface `Stack<T>` em duas versões: com **array** e com **lista ligada**
2. Aplicar a pilha na **avaliação de expressões pós-fixas**
3. Implementar a interface `Queue<T>` (FIFO) com lista ligada
4. Implementar o TAD **Union-Find** em duas versões (quick-find e quick-union ponderado)
5. Comparar desempenho entre implementações

## Como rodar Java no Colab

O Colab é um ambiente Python, mas a imagem padrão já vem com o **OpenJDK** instalado. Não precisamos trocar o kernel — usamos o seguinte padrão em todas as células:

| Passo | Ferramenta |
|-------|------------|
| Salvar uma classe Java | célula com `%%writefile NomeDaClasse.java` |
| Compilar tudo | célula com `!javac *.java` |
| Executar | célula com `!java NomeDaClasseComMain` |

> **Importante:** o nome da classe pública precisa bater com o nome do arquivo. Se você renomear, lembre-se de atualizar tanto o `%%writefile` quanto a declaração da classe.

## É para entregar ?

- Não, apenas faça uma Cópia para teu uso pessoal. Pode ser cobrado em prova o conteúdo desse colab.



---
## 0. Setup do ambiente

Verifique a instalação do Java e crie a pasta de trabalho.


In [ ]:
!java -version
!javac -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)
javac 17.0.18


In [ ]:
# Pasta de trabalho — todos os arquivos .java ficam aqui
import os, shutil
LAB = "/content/tad_lab"
if os.path.exists(LAB):
    shutil.rmtree(LAB)
os.makedirs(LAB)
%cd /content/tad_lab

/content/tad_lab


---
## Exercício 1 — `ArrayStack<T>` (15 min)

Implemente uma **pilha genérica baseada em array** que satisfaz a interface `Stack<T>`.

### Lembre-se

- Pilha é **LIFO**: o `pop` devolve o item inserido **mais recentemente**
- `T` é apagado em runtime (*type erasure*) — você **não pode** escrever `new T[N]`
- Idiom padrão: alocar `Object[]` e fazer **um único cast** `(T[]) new Object[maxN]` no construtor (com `@SuppressWarnings("unchecked")`)
- `push` e `pop` devem rodar em $O(1)$

### Tarefa

Complete os 4 TODOs em `ArrayStack.java`. A interface já está pronta abaixo.


In [ ]:
%%writefile Stack.java
public interface Stack<T> {
    boolean empty();
    void    push(T item);
    T       pop();
}

Writing Stack.java


In [ ]:
import java . util .*;

public class ArrayStack<T> implements Stack<T> {
    private T[] s; // T[] está generalizando um tipo generico para a criação da pilha na hora da criação obs: a classe é basicamente um array com metodos que simulam uma pilha.
    private int N; // cada operação de push ou pop ele aumenta, no primeiro caso, e diminui no ultimo caso.

    @SuppressWarnings("unchecked")
    public ArrayStack(int maxN) {
        // TODO 1: alocar o array s com capacidade maxN
        //         (use o idiom (T[]) new Object[maxN])
        s = (T[]) new Object[maxN];
        N = 0;
        // TODO 2: inicializar N (numero de elementos atuais)
    }

    @Override
    public boolean empty() {
        // TODO 3: retornar true sse a pilha esta vazia
        if(N == 0) {
            return true;
        }
        return false;
    }

    @Override
    public void push(T item) {
        // TODO 4: inserir item no topo
        s[N++] = item;
    }

    @Override
    public T pop() {
        // TODO 5: remover e retornar o item do topo
        if(N != 0){
            return s[--N];
        }
        return null;
    }
}


Writing ArrayStack.java


In [ ]:
%%writefile TestArrayStack.java
public class TestArrayStack {
    public static void main(String[] args) {
        Stack<String> s = new ArrayStack<>(10);
        check(s.empty(), "pilha nova deve estar vazia");

        s.push("L"); s.push("A"); s.push("S");
        check(!s.empty(), "depois de 3 push nao pode estar vazia");
        eq("S", s.pop(), "pop deve retornar o ultimo (LIFO)");
        eq("A", s.pop(), "proximo pop");

        s.push("T"); s.push("I");
        eq("I", s.pop(), "");
        eq("T", s.pop(), "");
        eq("L", s.pop(), "");
        check(s.empty(), "todos retirados, deve estar vazia");

        // Genericidade: outra parametrizacao
        Stack<Integer> n = new ArrayStack<>(5);
        n.push(1); n.push(2); n.push(3);
        eq(3, n.pop(), "Integer");
        eq(2, n.pop(), "Integer");

        System.out.println("OK -- ArrayStack passou em todos os testes");
    }

    static void check(boolean cond, String msg) {
        if (!cond) throw new AssertionError("FALHOU: " + msg);
    }
    static <T> void eq(T esp, T obt, String msg) {
        boolean ok = esp == null ? obt == null : esp.equals(obt);
        if (!ok) throw new AssertionError(
            "FALHOU: esperado <" + esp + ">, obtido <" + obt + "> — " + msg);
    }
}

Writing TestArrayStack.java


In [ ]:
!javac *.java && java TestArrayStack

Exception in thread "main" java.lang.AssertionError: FALHOU: pilha nova deve estar vazia
	at TestArrayStack.check(TestArrayStack.java:27)
	at TestArrayStack.main(TestArrayStack.java:4)


---
## Exercício 2 — `LinkedStack<T>` (15 min)

Mesma interface, agora com **lista ligada**. A vantagem: capacidade ilimitada (não precisa pré-alocar) e espaço proporcional ao uso real.

### Lembre-se

- A `head` da lista é o **topo** da pilha
- `push(x)`: cria um novo nó com `next = head`, depois faz `head = novoNo`
- `pop()`: salva `head.item`, faz `head = head.next`, devolve o item salvo
- Continua $O(1)$ tanto `push` quanto `pop`

### Tarefa

Complete os TODOs em `LinkedStack.java`.


In [ ]:
import java.util.*;

public class LinkedStack<T> implements Stack<T> {

    private class Node {
        T    item;
        Node next;
        Node(T item, Node next) {
            this.item = item;
            this.next = next;
        }
    }

    private Node head;

    public LinkedStack() {
        head = null;
    }

    @Override
    public boolean empty() {
        // TODO 1
        if(head == null){
            return true;
        }
        return false;
    }

    @Override
    public void push(T item) {
        // TODO 2: criar novo no apontando para head e atualizar head
        head = new Node(item, head); // observe que, o head sempre será o topo da lista, e ele sempre aponta para o objeto que era o head anterior, que antes era o topo.
    }

    @Override
    public T pop() {
        // TODO 3: salvar head.item, avancar head, retornar o salvo
        if(head != null) {
            T salvo = head.item;
            head = head.next; //head.next é um objeto, caso eu quisesse o item seria head.next.item
            return salvo;

        }
        return null;
    }
}

In [ ]:
%%writefile TestLinkedStack.java
public class TestLinkedStack {
    public static void main(String[] args) {
        Stack<String> s = new LinkedStack<>();
        check(s.empty(), "pilha nova vazia");

        for (String c : new String[]{"L","A","S","T","I"}) s.push(c);
        eq("I", s.pop(), "");
        eq("T", s.pop(), "");
        eq("S", s.pop(), "");
        eq("A", s.pop(), "");
        eq("L", s.pop(), "");
        check(s.empty(), "vazia ao final");

        // Stress: capacidade ilimitada
        Stack<Integer> big = new LinkedStack<>();
        int N = 100_000;
        for (int i = 0; i < N; i++) big.push(i);
        for (int i = N - 1; i >= 0; i--) eq(i, big.pop(), "stress i=" + i);
        check(big.empty(), "stress: vazia ao final");

        System.out.println("OK -- LinkedStack passou em todos os testes");
    }
    static void check(boolean cond, String msg) {
        if (!cond) throw new AssertionError("FALHOU: " + msg);
    }
    static <T> void eq(T esp, T obt, String msg) {
        boolean ok = esp == null ? obt == null : esp.equals(obt);
        if (!ok) throw new AssertionError(
            "FALHOU: esperado <" + esp + ">, obtido <" + obt + "> — " + msg);
    }
}

In [ ]:
!javac *.java && java TestLinkedStack

---
## Exercício 3 — Avaliação de expressões pós-fixas (15 min)

Use uma das suas pilhas para **avaliar uma expressão em notação pós-fixa** (RPN — Reverse Polish Notation).

### Regras

- A entrada é uma `String` com tokens separados por espaços (ex.: `"3 4 +"`)
- Tokens podem ser inteiros ou um dos operadores `+`, `-`, `*`, `/`
- Ao ler um operando: `push`
- Ao ler um operador: `pop` dois valores, aplica e faz `push` do resultado
- Atenção: o **primeiro** `pop` é o operando da **direita** ($b$), o **segundo** é o da **esquerda** ($a$). Para `-` e `/` isso importa.

### Exemplos

| Expressão | Equivale a | Resultado |
|-----------|------------|-----------|
| `"3 4 +"` | $3 + 4$ | `7` |
| `"5 1 2 + 4 * + 3 -"` | $5 + ((1+2) \times 4) - 3$ | `14` |
| `"10 2 /"` | $10 / 2$ | `5` |
| `"5 9 8 + 4 6 * * 7 + *"` | $5 \times (((9+8)\times(4\times6))+7)$ | `2075` |

### Tarefa

Complete `evaluate` em `Postfix.java`.


In [ ]:


public class Postfix {

    public static int evaluate(String expr) {
        Stack<Integer> s = new ArrayStack<>(1000);
        // TODO: percorrer cada token (use expr.split("\\s+"))
        //   - se for um numero: s.push(Integer.parseInt(t))
        //   - se for operador: pop b, pop a, push do resultado de (a op b)
        //   - cuidado com a ordem em '-' e '/'

        //criando a listas
        String[] expressao = expr.split("\\s+");

        //algoritmos logico:
        for(int i = 0; i < expressao.length; i++) {
            if(expressao[i].equals("+")) {
               s.push(s.pop() + s.pop());
            }
            else if(expressao[i].equals("*")) {
               s.push(s.pop() *s.pop());
            }
            else if(expressao[i].equals("-")) {
                int b = s.pop();
                int a = s.pop();
               s.push(a - b);
            }
            else if(expressao[i].equals("/")) {
                int b = s.pop();
                if(b == 0) {
                   s.push(0);
                }
                int a = s.pop();
               s.push(a / b);
            }
            else {
               s.push(Integer.parseInt(expressao[i]));
            }



        }


        return s.pop();
    }

    public static void main(String[] args) {
        System.out.println(evaluate("5 9 8 + 4 6 * * 7 + *"));
    }
}

In [ ]:
%%writefile TestPostfix.java
public class TestPostfix {
    public static void main(String[] args) {
        eq(7,    Postfix.evaluate("3 4 +"),                "soma simples");
        eq(14,   Postfix.evaluate("5 1 2 + 4 * + 3 -"),    "expressao com -");
        eq(5,    Postfix.evaluate("10 2 /"),               "divisao");
        eq(2075, Postfix.evaluate("5 9 8 + 4 6 * * 7 + *"),"exemplo do livro");
        eq(-1,   Postfix.evaluate("10 11 -"),              "subtracao nao comutativa");
        eq(0,    Postfix.evaluate("10 3 / 3 -"),           "divisao inteira");
        System.out.println("OK -- Postfix passou em todos os testes");
    }
    static void eq(int esp, int obt, String msg) {
        if (esp != obt) throw new AssertionError(
            "FALHOU: esperado " + esp + ", obtido " + obt + " — " + msg);
    }
}

In [ ]:
!javac *.java && java TestPostfix

---
## Exercício 4 — `LinkedQueue<T>` (15 min)

Agora a fila **FIFO**: insere no `tail`, remove do `head`. Use lista ligada com **dois** ponteiros.

### Lembre-se

- `put(x)` insere no final (`tail`)
- `get()` remove do início (`head`)
- **Caso especial**: quando a fila estava vazia, o `put` precisa atualizar **head e tail** para o novo nó
- Continua $O(1)$ tanto `put` quanto `get`

### Tarefa

Complete os TODOs em `LinkedQueue.java`.


In [ ]:
%%writefile Queue.java
public interface Queue<T> {
    boolean empty();
    void    put(T item);   // insere no final (tail)
    T       get();         // remove do inicio (head)
}

In [ ]:
import java.util.*;

public class LinkedQueue<T> implements Queue<T> {

    private class Node {
        T item;
        Node next;
        Node(T item) {
            this.item = item;
            this.next = null;
        }
    }

    private Node head, tail;

    public LinkedQueue() {
        head = null;
        tail = null;
    }

    @Override
    public boolean empty() {
        // TODO 1
        if(head == null) {
            return true;
        }
        return false;
    }

    @Override
    public void put(T item) {
        // TODO 2:
        //   - guarde o tail antigo
        //   - tail = novo Node(item)
        //   - se a fila estava vazia, head = tail
        //   - senao, antigoTail.next = tail
        Node oldTail = tail;
        tail = new Node(item);
        if(empty()){
            head = tail;
        } else {
            oldTail.next = tail;
        }

    }

    @Override
    public T get() {
        // TODO 3:
        //   - salvar head.item
        //   - head = head.next
        //   - retornar o item salvo
        //   (nao se preocupe com tail aqui)
        if(!(empty())) {
            T obej = head.item;
            head = head.next;
            return obej;
        }
        return null;
    }
}

In [ ]:
%%writefile TestLinkedQueue.java
public class TestLinkedQueue {
    public static void main(String[] args) {
        Queue<String> q = new LinkedQueue<>();
        check(q.empty(), "fila nova deve estar vazia");

        q.put("F"); q.put("I"); q.put("R"); q.put("S"); q.put("T");
        check(!q.empty(), "depois de puts nao pode estar vazia");
        eq("F", q.get(), "FIFO: primeiro a entrar, primeiro a sair");
        eq("I", q.get(), "");
        eq("R", q.get(), "");
        eq("S", q.get(), "");
        eq("T", q.get(), "");
        check(q.empty(), "vazia ao final");

        // Intercalando put/get para verificar o caso de fila esvaziada
        q.put("A"); eq("A", q.get(), "put-get unico");
        check(q.empty(), "vazia depois");
        q.put("B"); q.put("C"); eq("B", q.get(), "");
        q.put("D"); eq("C", q.get(), ""); eq("D", q.get(), "");

        System.out.println("OK -- LinkedQueue passou em todos os testes");
    }
    static void check(boolean cond, String msg) {
        if (!cond) throw new AssertionError("FALHOU: " + msg);
    }
    static <T> void eq(T esp, T obt, String msg) {
        boolean ok = esp == null ? obt == null : esp.equals(obt);
        if (!ok) throw new AssertionError(
            "FALHOU: esperado <" + esp + ">, obtido <" + obt + "> — " + msg);
    }
}

In [ ]:
!javac *.java && java TestLinkedQueue

---
## Exercício 5 — Union-Find: versão `Quick-Find` (10 min)

A versão mais ingênua do Union-Find:

- `id[i]` = identificador do componente do elemento `i`
- **Inicialmente** cada elemento está no seu próprio componente: `id[i] = i`
- **`find(p, q)`**: $p$ e $q$ estão conectados sse `id[p] == id[q]` — $O(1)$
- **`unite(p, q)`**: troca **todos** os elementos com `id[p]` para `id[q]` — $O(N)$ no pior caso

### Tarefa

Complete `QuickFindUF.java`.


In [5]:
%%writefile QuickFindUF.java
public class QuickFindUF {
    private int[] id;

    public QuickFindUF(int N) {
        id = new int[N];
        for (int i = 0; i < N; i++) id[i] = i;
    }

    public boolean find(int p, int q) {
        // TODO 1
        if(id[p] == id[q]) {
            return true;
        }
        return false;
    }

    public void unite(int p, int q) {
        // TODO 2:
        //   - se ja conectados, nao faz nada
        int idp = id[p];
        int idq = id[q];

        if (find(idp, idq)) {
            return;
        }
        //   - senao, percorre o array e troca todo id[i] == id[p] para id[q]
        for(int i = 0; i < id.length; i++ ) {
            if(id[i] == idp) {
                id[i] = idq;
            }
        }
    }
}

Writing QuickFindUF.java


In [4]:
%%writefile TestQuickFind.java
public class TestQuickFind {
    public static void main(String[] args) {
        // cenario do slide: 6 elementos
        QuickFindUF uf = new QuickFindUF(6);

        check(!uf.find(0, 1), "0 e 1 separados no inicio");
        uf.unite(0, 1);
        check(uf.find(0, 1), "0 e 1 unidos");

        uf.unite(2, 3);
        check(uf.find(2, 3), "2 e 3 unidos");
        check(!uf.find(0, 2), "{0,1} e {2,3} ainda separados");

        uf.unite(0, 3);
        check(uf.find(1, 2), "transitividade: 1-0-3-2");
        check(!uf.find(1, 5), "5 ainda isolado");

        uf.unite(4, 5);
        uf.unite(3, 5);
        for (int i = 0; i < 6; i++)
            for (int j = 0; j < 6; j++)
                check(uf.find(i, j), "tudo conectado: " + i + "-" + j);

        System.out.println("OK -- QuickFindUF passou em todos os testes");
    }
    static void check(boolean cond, String msg) {
        if (!cond) throw new AssertionError("FALHOU: " + msg);
    }
}

Overwriting TestQuickFind.java


In [6]:
!javac *.java && java TestQuickFind

OK -- QuickFindUF passou em todos os testes


---
## Exercício 6 — Union-Find: `Quick-Union Ponderado` (20 min)

A versão eficiente, do slide. Em vez de array de IDs, mantemos uma **floresta**:

- `id[i]` = pai de `i`. A raiz é o representante do componente
- `sz[i]` = tamanho da árvore enraizada em `i` (só faz sentido se `i` é raiz)
- **`find(p)` privado**: sobe na árvore até a raiz (`while (x != id[x]) x = id[x]`)
- **`find(p, q)` público**: as raízes coincidem?
- **`unite(p, q)`**: liga a raiz da árvore **menor** sob a raiz da **maior**, atualizando `sz`. Isso mantém a altura em $O(\log N)$.

### Tarefa

Complete `WeightedQuickUnionUF.java` e veja a comparação de desempenho.


In [7]:
%%writefile WeightedQuickUnionUF.java
public class WeightedQuickUnionUF {
    private int[] id, sz;

    public WeightedQuickUnionUF(int N) {
        id = new int[N];
        sz = new int[N];
        for (int i = 0; i < N; i++) {
            id[i] = i;
            sz[i] = 1;
        }
    }

    private int find(int x) {
        // TODO 1: subir enquanto x != id[x]
        while(x != id[x]){
            x = id[x];
        }
        return x;
    }

    public boolean find(int p, int q) {
        // TODO 2: as raizes coincidem?
        if(find(q) == find(p)){ // condição para saber se eles estão apontando para mesmo "pai"
            return true;
        }
        return false;
    }

    public void unite(int p, int q) {
        // TODO 3:
        //   - i = raiz de p, j = raiz de q
        int i = find(p);
        int j = find(q);
        //   - se i == j, nada a fazer
        //   - se sz[i] < sz[j]: id[i] = j; sz[j] += sz[i]
        //   - senao:            id[j] = i; sz[i] += sz[j]
        if (i == j) {
            return;
        }
        if (sz[i] < sz[j]) {
            id[i] = j;
            sz[j] += sz[i];
        } else {
            id[j] = i;
            sz[i] += sz[j];
        }
    }
}

Writing WeightedQuickUnionUF.java


In [8]:
%%writefile TestWQU.java
public class TestWQU {
    public static void main(String[] args) {
        // Mesmos testes funcionais do QuickFind
        WeightedQuickUnionUF uf = new WeightedQuickUnionUF(6);
        check(!uf.find(0, 1), "");
        uf.unite(0, 1); check(uf.find(0, 1), "0-1");
        uf.unite(2, 3); check(uf.find(2, 3), "2-3");
        uf.unite(0, 3); check(uf.find(1, 2), "transitividade");
        check(!uf.find(1, 5), "5 isolado");
        uf.unite(4, 5); uf.unite(3, 5);
        for (int i = 0; i < 6; i++)
            for (int j = 0; j < 6; j++)
                check(uf.find(i, j), "tudo conectado: " + i + "-" + j);

        // Comparacao de desempenho com QuickFind
        long t1 = bench(true,  100_000);
        long t2 = bench(false, 100_000);
        System.out.printf("WeightedQuickUnion: %d ms%n", t1);
        System.out.printf("QuickFind         : %d ms%n", t2);
        System.out.printf("Speedup           : %.1fx%n", (double) t2 / Math.max(1, t1));

        System.out.println("OK -- WeightedQuickUnionUF passou em todos os testes");
    }

    static long bench(boolean weighted, int N) {
        java.util.Random r = new java.util.Random(42);
        long t0 = System.currentTimeMillis();
        if (weighted) {
            WeightedQuickUnionUF uf = new WeightedQuickUnionUF(N);
            for (int k = 0; k < N; k++) uf.unite(r.nextInt(N), r.nextInt(N));
        } else {
            QuickFindUF uf = new QuickFindUF(N);
            for (int k = 0; k < N; k++) uf.unite(r.nextInt(N), r.nextInt(N));
        }
        return System.currentTimeMillis() - t0;
    }

    static void check(boolean cond, String msg) {
        if (!cond) throw new AssertionError("FALHOU: " + msg);
    }
}

Writing TestWQU.java


In [9]:
!javac *.java && java TestWQU

WeightedQuickUnion: 26 ms
QuickFind         : 9026 ms
Speedup           : 347.2x
OK -- WeightedQuickUnionUF passou em todos os testes


---
## Desafio integrador (opcional, 15 min)

Use o `WeightedQuickUnionUF` para responder, em **uma única passagem**, ao problema de **conectividade dinâmica**:

> Dada uma sequência de pares $(p, q)$, imprima apenas aqueles que **conectam novos componentes** (ou seja, ignore pares cujos extremos já estão no mesmo componente). Ao final, informe quantos componentes restam.

Esta é exatamente a forma como o algoritmo de Kruskal usa o Union-Find para evitar ciclos na árvore geradora mínima.

### Tarefa

Complete `Conectividade.java`.


In [ ]:
%%writefile Conectividade.java
public class Conectividade {

    public static int processa(int N, int[][] pares) {
        WeightedQuickUnionUF uf = new WeightedQuickUnionUF(N);
        int componentes = N;
        for (int[] par : pares) {
            int p = par[0], q = par[1];
            // TODO:
            //   - se p e q ja estao conectados, ignore (continue)
            //   - senao: uf.unite(p, q); imprima "p-q"; decremente componentes
        }
        return componentes;
    }

    public static void main(String[] args) {
        int[][] entrada = {
            {3,4}, {4,9}, {8,0}, {2,3}, {5,6}, {2,9},
            {5,9}, {7,3}, {4,8}, {5,6}, {0,2}, {6,1}
        };
        int restantes = processa(10, entrada);
        System.out.println("Componentes restantes: " + restantes);
    }
}

In [ ]:
!javac *.java && java Conectividade

**Saída esperada:**

```
3-4
4-9
8-0
2-3
5-6
5-9
7-3
4-8
6-1
Componentes restantes: 1
```

Note que o par `2-9` é **ignorado** (ambos já no componente `{2, 3, 4, 9}` formado por `2-3, 3-4, 4-9`), bem como `5-6, 0-2`. O par `5-9` **funde** os componentes `{0, 2, 3, 4, 7, 8, 9}` e `{5, 6}` num único grande, e em seguida `6-1` puxa o `1` para dentro — sobra **um único componente** com todos os 10 elementos.

> Atenção: o desenho do slide *Conectividade em Redes* mostra dois componentes finais, mas isso não é coerente com a entrada listada — se você seguir os pares na ordem dada, `5-9` conecta as duas regiões. Aqui usamos a regra real do algoritmo.


---
## Soluções de referência

> **Não olhe antes de tentar!** As soluções estão abaixo apenas para conferência depois que você concluir.

Para usar uma solução, descomente o `%%writefile` da célula correspondente, execute-a, e depois rode novamente a célula de teste daquela seção.


### ArrayStack — solução

In [ ]:
# %%writefile ArrayStack.java
public class ArrayStack<T> implements Stack<T> {
    private T[] s;
    private int N;

    @SuppressWarnings("unchecked")
    public ArrayStack(int maxN) {
        s = (T[]) new Object[maxN];
        N = 0;
    }
    public boolean empty()        { return N == 0; }
    public void    push(T item)   { s[N++] = item; }
    public T       pop()          { return s[--N]; }
}

### LinkedStack — solução

In [ ]:
# %%writefile LinkedStack.java
public class LinkedStack<T> implements Stack<T> {
    private class Node {
        T item; Node next;
        Node(T item, Node next) { this.item = item; this.next = next; }
    }
    private Node head;
    public LinkedStack() { head = null; }
    public boolean empty()      { return head == null; }
    public void    push(T item) { head = new Node(item, head); }
    public T       pop() {
        T v = head.item;
        head = head.next;
        return v;
    }
}

### Postfix — solução

In [ ]:
# %%writefile Postfix.java
public class Postfix {
    public static int evaluate(String expr) {
        Stack<Integer> s = new LinkedStack<>();
        for (String t : expr.trim().split("\\s+")) {
            switch (t) {
                case "+": { int b = s.pop(), a = s.pop(); s.push(a + b); break; }
                case "-": { int b = s.pop(), a = s.pop(); s.push(a - b); break; }
                case "*": { int b = s.pop(), a = s.pop(); s.push(a * b); break; }
                case "/": { int b = s.pop(), a = s.pop(); s.push(a / b); break; }
                default:  { s.push(Integer.parseInt(t)); }
            }
        }
        return s.pop();
    }
    public static void main(String[] args) {
        System.out.println(evaluate("5 9 8 + 4 6 * * 7 + *"));
    }
}

### LinkedQueue — solução

In [ ]:
# %%writefile LinkedQueue.java
public class LinkedQueue<T> implements Queue<T> {
    private class Node {
        T item; Node next;
        Node(T item) { this.item = item; this.next = null; }
    }
    private Node head, tail;
    public LinkedQueue() { head = null; tail = null; }
    public boolean empty() { return head == null; }
    public void put(T item) {
        Node t = tail;
        tail = new Node(item);
        if (empty()) head = tail;
        else t.next = tail;
    }
    public T get() {
        T v = head.item;
        head = head.next;
        if (head == null) tail = null;
        return v;
    }
}

### QuickFindUF — solução

In [ ]:
# %%writefile QuickFindUF.java
public class QuickFindUF {
    private int[] id;
    public QuickFindUF(int N) {
        id = new int[N];
        for (int i = 0; i < N; i++) id[i] = i;
    }
    public boolean find(int p, int q) { return id[p] == id[q]; }
    public void unite(int p, int q) {
        int t = id[p];
        if (t == id[q]) return;
        for (int i = 0; i < id.length; i++)
            if (id[i] == t) id[i] = id[q];
    }
}

### WeightedQuickUnionUF — solução

In [ ]:
# %%writefile WeightedQuickUnionUF.java
public class WeightedQuickUnionUF {
    private int[] id, sz;
    public WeightedQuickUnionUF(int N) {
        id = new int[N]; sz = new int[N];
        for (int i = 0; i < N; i++) { id[i] = i; sz[i] = 1; }
    }
    private int find(int x) {
        while (x != id[x]) x = id[x];
        return x;
    }
    public boolean find(int p, int q) { return find(p) == find(q); }
    public void unite(int p, int q) {
        int i = find(p), j = find(q);
        if (i == j) return;
        if (sz[i] < sz[j]) { id[i] = j; sz[j] += sz[i]; }
        else               { id[j] = i; sz[i] += sz[j]; }
    }
}

### Conectividade — solução

In [ ]:
# %%writefile Conectividade.java
public class Conectividade {
    public static int processa(int N, int[][] pares) {
        WeightedQuickUnionUF uf = new WeightedQuickUnionUF(N);
        int componentes = N;
        for (int[] par : pares) {
            int p = par[0], q = par[1];
            if (uf.find(p, q)) continue;
            uf.unite(p, q);
            System.out.println(p + "-" + q);
            componentes--;
        }
        return componentes;
    }
    public static void main(String[] args) {
        int[][] entrada = {
            {3,4}, {4,9}, {8,0}, {2,3}, {5,6}, {2,9},
            {5,9}, {7,3}, {4,8}, {5,6}, {0,2}, {6,1}
        };
        int restantes = processa(10, entrada);
        System.out.println("Componentes restantes: " + restantes);
    }
}

---
## Para refletir (entregável)

Responda em uma célula de markdown ao final do seu notebook:

1. **Generics:** por que escrevemos `Stack<T>` em vez de `Stack` que armazena `Object`? Cite ao menos duas vantagens concretas.
2. **Array vs. Lista:** em que cenário você preferiria `ArrayStack` em detrimento de `LinkedStack`? E o contrário?
3. **Quick-Find vs. Quick-Union:** olhando os tempos da célula de benchmark, aproximadamente quantas vezes mais rápido foi o weighted quick-union? Por quê?
4. **Postfix:** por que o algoritmo nunca precisa de parênteses na entrada? O que aconteceria se um operador chegasse com a pilha contendo menos de dois elementos?
